<a href="https://colab.research.google.com/github/garaimg/repositorios-git-pruebas/blob/main/ds4_dwarfstar_explicado_y_probado.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DwarfStar (`ds4`): qué es, para qué sirve y validación en Google Colab

Repositorio: <https://github.com/antirez/ds4> · Autor: Salvatore Sanfilippo (*antirez*, creador de Redis)

## 1. Qué es

Un LLM tiene dos partes: los **pesos** (un archivo enorme con los números que el modelo aprendió al entrenarse)
y el **motor de inferencia** (el programa que carga esos pesos y hace los cálculos para generar texto, token
a token).

`ds4` es un **motor de inferencia escrito en C**, sin Python ni PyTorch. Su objetivo es ejecutar unos pocos
modelos muy grandes de pesos abiertos en equipos que una persona puede comprar:
**DeepSeek V4 Flash / V4.1 Flash / V4 PRO**, **GLM 5.2 / 5.3** y **Qwen3.8 Flash Next**.

**¿En qué se diferencia de llama.cpp u Ollama?** Esos proyectos son genéricos: cargan casi cualquier modelo en
formato GGUF. `ds4` hace lo contrario: soporta **pocos modelos, pero optimizados uno a uno**. Por eso solo acepta
los archivos GGUF que publica el propio proyecto. El autor reconoce que se apoya en el trabajo de llama.cpp
(reutiliza sus formatos de cuantización y algunos kernels).

**Dónde se ejecuta** (lo que el proyecto llama *backends*):

| Backend | Hardware |
|---|---|
| Metal | GPU de los Mac con Apple Silicon (el objetivo principal) |
| CUDA | GPUs NVIDIA, sobre todo la DGX Spark; también varias GPUs a la vez |
| ROCm | GPUs AMD, en concreto los PC con Strix Halo |
| CPU | Solo como referencia para comprobar que las GPUs calculan bien (es muy lento) |

## 2. Para qué sirve

Con el modelo descargado y el hardware adecuado, el repositorio ofrece cinco programas:

| Programa | Para qué sirve |
|---|---|
| `ds4` | Chatear con el modelo en la terminal o lanzar una pregunta suelta (`-p "..."`) |
| `ds4-server` | Servidor HTTP **compatible con las APIs de OpenAI y Anthropic**: cualquier aplicación que hable con ChatGPT o Claude puede usar el modelo local cambiando solo la URL |
| `ds4-agent` | Agente de programación (similar a Claude Code o Codex) que lee y edita ficheros y ejecuta comandos usando el modelo local |
| `ds4-eval` | Evalúa la calidad del modelo con preguntas de *benchmarks* reales (GPQA, AIME 2025, SuperGPQA) |
| `ds4-bench` | Mide la velocidad en tokens por segundo |

Funciones avanzadas destacables:

- **SSD streaming**: si el modelo no cabe en la RAM, va leyendo del SSD las partes que necesita en cada momento.
- **Inferencia distribuida**: repartir el modelo entre dos o más equipos (por ejemplo, dos Mac conectados) para
  sumar su memoria.
- **Decodificación especulativa**: el modelo propone varios tokens de golpe y luego los verifica, lo que
  acelera la generación en algunos casos.
- **Visión**: algunos modelos aceptan imágenes además de texto.

**¿Por qué usarlo?** Privacidad (los datos no salen del equipo), coste cero por uso una vez tienes el hardware
y la posibilidad de estudiar un motor de inferencia real y relativamente compacto.

## 3. Conceptos clave

**Mixture-of-Experts (MoE).** DeepSeek V4 Flash tiene **284.000 millones de parámetros** (284B), pero no los usa todos
a la vez. Está dividido en muchos "expertos" (pequeñas redes) y, para cada token, un componente llamado *router*
elige solo unos pocos. Así, en cada paso trabajan unos **13B** parámetros. Consecuencia: la velocidad depende de
los 13B activos, pero la **memoria** tiene que albergar los 284B, porque cualquier experto puede hacer falta
en el siguiente token.

**Cuantización.** Consiste en guardar cada parámetro con menos bits. Los números se agrupan en bloques que
comparten un factor de escala, y así cada número individual ocupa muy poco. Se pierde algo de precisión a
cambio de reducir mucho el tamaño:

| Formato | Bits por parámetro | Idea |
|---|---:|---|
| BF16 | 16 | Sin cuantizar (precisión original) |
| `Q8_0` | 8,5 | Enteros de 8 bits, casi sin pérdida |
| `Q4_K` | 4,5 | 4 bits por número; un estándar habitual en llama.cpp |
| `MXFP4` | 4,25 | 4 bits en coma flotante; el formato en que DeepSeek publica sus expertos |
| `Q2_K` / `IQ2_XXS` | ~2–2,6 | Compresión extrema; solo funciona bien en partes del modelo que la toleran |

El modelo Flash Q2 de `ds4` aplica la compresión extrema **solo a los expertos** y deja con más precisión las
partes más delicadas (atención, experto compartido, capa de salida).

**Atención comprimida y KV cache.** Para no recalcular toda la conversación en cada token, el modelo guarda
información de los tokens anteriores: la **KV cache**, su "memoria de trabajo". Con contextos muy largos esa
caché crece muchísimo. DeepSeek V4 la comprime: mantiene completos solo los últimos 128 tokens y resume la
historia anterior agrupando tokens de 4 en 4 o de 128 en 128, según la capa. Gracias a eso admite contextos de
hasta **1 millón de tokens**.

Cálculo de cuánto ocupa el modelo según el formato:

In [1]:
# Tamaño de DeepSeek V4 Flash (284B parámetros) según los bits por parámetro
P_TOTAL, P_ACTIVOS = 284e9, 13e9
formatos = {"BF16": 16, "Q8_0": 8.5, "Q4_K": 4.5, "MXFP4": 4.25, "Q2_K": 2.625, "IQ2_XXS": 2.0625}

print(f"{'Formato':10s} {'bits/peso':>9s} {'pesos totales':>14s} {'leído por token (13B activos)':>30s}")
for nombre, bpw in formatos.items():
    total = P_TOTAL * bpw / 8 / 2**30
    activo = P_ACTIVOS * bpw / 8 / 2**30
    print(f"{nombre:10s} {bpw:9.4f} {total:11.0f} GiB {activo:27.1f} GiB")

# Bits/peso medios implícitos en el GGUF oficial Flash Q2 (~81 GiB según docs/MODELS.md)
print(f"\nGGUF Flash Q2 (~81 GiB) ⇒ {81 * 2**30 * 8 / P_TOTAL:.2f} bits/peso de media "
      "(expertos a ~2 bits y el resto con más precisión)")

Formato    bits/peso  pesos totales  leído por token (13B activos)
BF16         16.0000         529 GiB                        24.2 GiB
Q8_0          8.5000         281 GiB                        12.9 GiB
Q4_K          4.5000         149 GiB                         6.8 GiB
MXFP4         4.2500         141 GiB                         6.4 GiB
Q2_K          2.6250          87 GiB                         4.0 GiB
IQ2_XXS       2.0625          68 GiB                         3.1 GiB

GGUF Flash Q2 (~81 GiB) ⇒ 2.45 bits/peso de media (expertos a ~2 bits y el resto con más precisión)


**¿Qué limita la velocidad?** Hay dos fases:

- ***Prefill*** (leer tu mensaje): se procesan miles de tokens a la vez, así que cada parámetro leído de memoria
  se reaprovecha muchas veces. Aquí manda la potencia de cálculo de la GPU.
- ***Decode*** (generar la respuesta): se genera un token detrás de otro y para cada uno hay que leer de memoria
  los ~13B parámetros activos. Aquí manda la **velocidad de la memoria** (ancho de banda).

Por eso se puede estimar un máximo teórico: `tokens/s ≤ ancho de banda / bytes leídos por token`.

In [2]:
# Velocidad máxima teórica de generación frente a las medidas publicadas en docs/PERFORMANCE.md
bpw_activos = 3.0          # suposición: media de ~3 bits por parámetro activo (expertos a ~2 bits, resto a 8)
bytes_token = P_ACTIVOS * bpw_activos / 8

medido = {  # Flash Q2, contexto de 2048 tokens (docs/PERFORMANCE.md)
    "M5 Max 128 GB":     {"prefill": 790.18, "decode": 39.35},
    "DGX Spark 128 GB":  {"prefill": 825.76, "decode": 18.05},
}
for bw_gbs in (273, 546):  # dos velocidades de memoria de ejemplo, en GB/s
    print(f"Con memoria de {bw_gbs} GB/s → máximo teórico ≈ {bw_gbs * 1e9 / bytes_token:5.0f} tokens/s "
          f"({bytes_token / 1e9:.1f} GB leídos por token)")
print()
for maquina, m in medido.items():
    print(f"{maquina:18s} medido: lectura (prefill) {m['prefill']:7.1f} t/s · generación (decode) {m['decode']:5.1f} t/s "
          f"(prefill/decode ≈ {m['prefill'] / m['decode']:.0f}×)")
print("\nEl máximo teórico ignora otros costes (leer la KV cache, sincronizaciones, etc.),"
      " por eso la velocidad real queda por debajo.")

Con memoria de 273 GB/s → máximo teórico ≈    56 tokens/s (4.9 GB leídos por token)
Con memoria de 546 GB/s → máximo teórico ≈   112 tokens/s (4.9 GB leídos por token)

M5 Max 128 GB      medido: lectura (prefill)   790.2 t/s · generación (decode)  39.4 t/s (prefill/decode ≈ 20×)
DGX Spark 128 GB   medido: lectura (prefill)   825.8 t/s · generación (decode)  18.1 t/s (prefill/decode ≈ 46×)

El máximo teórico ignora otros costes (leer la KV cache, sincronizaciones, etc.), por eso la velocidad real queda por debajo.


## 4. Limitación de Colab: no se puede ejecutar el modelo

El modelo más pequeño que soporta (DeepSeek V4 Flash Q2) ocupa **unos 81 GiB**, y el proyecto lo recomienda para
equipos con **96–128 GB** de memoria (más un margen para la KV cache). Colab gratuito ofrece unos 12 GB de RAM y
15 GB de memoria en la GPU, y el SSD streaming tampoco está disponible para CPU.

In [3]:
import os, shutil, subprocess

ram_gb = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 2**30
disco_libre_gb = shutil.disk_usage("/content").free / 2**30
gpu = subprocess.run("nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv,noheader",
                     shell=True, capture_output=True, text=True).stdout.strip() or "ninguna"

print(f"RAM de este Colab        : {ram_gb:5.1f} GiB")
print(f"Disco libre              : {disco_libre_gb:5.1f} GiB")
print(f"GPU                      : {gpu}")
print(f"\nModelo más pequeño (Q2)  : ~81 GiB  → ¿cabe en RAM? {'sí' if ram_gb > 96 else 'NO ❌'}")

RAM de este Colab        :  12.7 GiB
Disco libre              : 235.6 GiB
GPU                      : Tesla T4, 15360 MiB, 7.5

Modelo más pequeño (Q2)  : ~81 GiB  → ¿cabe en RAM? NO ❌


Por eso este notebook comprueba **todo lo que se puede verificar sin el modelo**:

1. Que el código **compila** en Linux sin errores.
2. Que los programas **arrancan** y, sin modelo, fallan de forma controlada.
3. Que pasan los **tests automáticos que no necesitan el modelo**, los mismos que el autor ejecuta con
   `make test`.
4. Si hay GPU, que el código **CUDA compila** para la tarjeta de Colab.

Al final, una tabla resume cada prueba como ✅ / ⚠️ / ❌.

## 5. Preparación

Una función auxiliar que ejecuta cada comando, muestra el final de su salida y apunta si ha ido bien para la tabla final.

In [4]:
import time, re
from IPython.display import display, Markdown

REPO_DIR = "/content/ds4"
RESULTADOS = []

def ejecutar(nombre, comando, que_comprueba="", mostrar_lineas=15, aviso_si=None):
    # Ejecuta un comando de terminal, enseña el final de la salida y apunta si ha ido bien.
    # aviso_si: texto que, si aparece en la salida de un fallo, lo convierte en ⚠️ (problema conocido del entorno).
    t0 = time.time()
    p = subprocess.run(comando, shell=True, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    salida = p.stdout.strip().splitlines()
    print(f"▶ {nombre}\n  $ {comando}")
    if que_comprueba:
        print(f"  ℹ️  {que_comprueba}")
    for linea in salida[-mostrar_lineas:]:
        print("   ", linea)
    if p.returncode == 0:
        estado = "✅ OK"
    elif aviso_si and any(aviso_si in l for l in salida):
        estado = "⚠️ Aviso"
    else:
        estado = "❌ FALLO"
    print(f"  → {estado}  ({time.time()-t0:.1f} s)\n")
    RESULTADOS.append({"Prueba": nombre, "Resultado": estado, "Qué comprueba": que_comprueba,
                       "Tiempo (s)": round(time.time() - t0, 1)})
    return p.returncode == 0

## 6. Descargar el código

No hay `requirements.txt` ni `pip install`: al estar escrito en C, solo necesita un compilador (`gcc`) y `make`,
que ya vienen en Colab.

In [5]:
if not os.path.exists(REPO_DIR):
    !git clone -q --depth 1 https://github.com/antirez/ds4 {REPO_DIR}
%cd {REPO_DIR}
!git log -1 --format="Versión probada: commit %h del %ad — %s" --date=short
!echo "Líneas de código C del motor principal (ds4.c):" && wc -l < ds4.c

/content/ds4
Versión probada: commit b3f5018 del 2026-11-03 — Read half-aligned Qwen Q2 weight rows correctly on Metal
Líneas de código C del motor principal (ds4.c):
89775


## 7. Compilar

El `Makefile` tiene un objetivo para cada plataforma (`make` en Mac, `make cuda-spark`, `make strix-halo`...).
En Linux sin GPU usamos **`make cpu`**, que compila los cinco programas con el backend de CPU. No sirve para usar
el modelo en serio (sería lentísimo), pero sí para comprobar que todo el código compila.

In [6]:
ejecutar("Compilar con `make cpu`", "make -j$(nproc) cpu 2>&1 | tail -3 && ls -lh ds4 ds4-server ds4-agent ds4-eval ds4-bench",
         "Que el código compila y genera los 5 programas")

▶ Compilar con `make cpu`
  $ make -j$(nproc) cpu 2>&1 | tail -3 && ls -lh ds4 ds4-server ds4-agent ds4-eval ds4-bench
  ℹ️  Que el código compila y genera los 5 programas
    cc -O3 -ffast-math -g -march=native -Wall -Wextra -std=c99 -D_GNU_SOURCE -fno-finite-math-only -o ds4-bench ds4_bench_cpu.o ds4_help.o ds4_gpu_args_cpu.o ds4_cpu.o ds4_image.o ds4_distributed.o ds4_tp.o ds4_ssd.o ds4_layer_pack.o -lm -pthread
    cc -O3 -ffast-math -g -march=native -Wall -Wextra -std=c99 -D_GNU_SOURCE -fno-finite-math-only -o ds4-eval ds4_eval_cpu.o ds4_eval_cases.o ds4_help.o ds4_cpu.o ds4_image.o ds4_distributed.o ds4_tp.o ds4_ssd.o ds4_layer_pack.o -lm -pthread
    cc -O3 -ffast-math -g -march=native -Wall -Wextra -std=c99 -D_GNU_SOURCE -fno-finite-math-only -o ds4-agent ds4_agent_cpu.o ds4_help.o ds4_prompt_prefix.o ds4_web.o ds4_kvstore.o linenoise.o ds4_gpu_args_cpu.o ds4_cpu.o ds4_image.o ds4_distributed.o ds4_tp.o ds4_ssd.o ds4_layer_pack.o -lm -pthread
    -rwxr-xr-x 1 root root 4.1M Nov

True

## 8. ¿Arrancan los programas?

Comprobación rápida (*smoke test*): que los programas arrancan, muestran su ayuda y, si no encuentran el modelo,
dan un error claro en vez de bloquearse.

In [7]:
ejecutar("`ds4 --help`", "./ds4 --help | head -20", "Que el programa principal arranca y muestra sus opciones")
ejecutar("`ds4` sin modelo da error controlado",
         "./ds4 -p 'hola' 2>&1; test $? -ne 0 && echo '(ha fallado como debe: no hay modelo)'",
         "Que sin el modelo da un error claro en vez de bloquearse")
ejecutar("`ds4-server --help`", "./ds4-server --help | head -8", "Que el servidor compatible con OpenAI/Anthropic arranca")

▶ `ds4 --help`
  $ ./ds4 --help | head -20
  ℹ️  Que el programa principal arranca y muestra sus opciones
    Model And Runtime
      -m, --model FILE               GGUF model path. Default: ds4flash.gguf
      --vision FILE                  Vision encoder GGUF for the selected model.
      --metal | --cuda | --cpu       Select the backend explicitly.
      --backend NAME                 Backend name: metal, cuda, or cpu.
      --gpu-vram N[,N,...]|auto      CUDA VRAM budgets per device, in GiB, or auto-detect free VRAM.
      --gpu-devices N[,N,...]        CUDA device indices used by multi-GPU placement.
      --cuda-tensor-parallel         Enable the paired DeepSeek tensor/expert path on an even multi-GPU CUDA placement.
      -c, --ctx N                    Allocated context tokens.
      -t, --threads N                CPU helper threads for host-side/reference work.
      --power N                      GPU duty-cycle target, 1..100. Default: 100
      --ssd-streaming                

True

### Las preguntas de evaluación de `ds4-eval`

`ds4-eval` incluye preguntas de *benchmarks* conocidos: física de nivel doctorado (**GPQA Diamond**), problemas de
olimpiada matemática (**AIME 2025**) y preguntas de muchas áreas (**SuperGPQA**). Sirven para comprobar que el
modelo sigue respondiendo bien tras un cambio en el motor. Sin cargar el modelo se puede listar el banco,
validar que está bien formado y probar el código que extrae la respuesta final ("C", "70"...) del texto generado.

In [8]:
ejecutar("Listar preguntas de `ds4-eval`", "./ds4-eval --list-cases | head -12; echo ...; echo \"Total: $(./ds4-eval --list-cases | tail -n +2 | wc -l) preguntas\"",
         "Que el banco de preguntas está incluido y se puede listar", mostrar_lineas=20)
ejecutar("Validar las preguntas de `ds4-eval`", "./ds4-eval --validate-cases",
         "Que todas las preguntas y respuestas esperadas están bien formadas")
ejecutar("Corrector automático de respuestas", "./ds4-eval --self-test-extractors",
         "Que se extrae bien la respuesta final del texto del modelo")

▶ Listar preguntas de `ds4-eval`
  $ ./ds4-eval --list-cases | head -12; echo ...; echo "Total: $(./ds4-eval --list-cases | tail -n +2 | wc -l) preguntas"
  ℹ️  Que el banco de preguntas está incluido y se puede listar
    #	source	domain	id	answer-kind	title
    1	GPQA Diamond	Physics	recNu3MXkvWUzHZr9	choice	An intelligent civilization in the Large Magellanic Cloud has
    2	SuperGPQA	Agronomy	001b51d76b4d422988f2c11f104a2c6c	choice	In production, grass powder is often pressed into pellets , and
    3	AIME2025	Number Theory	aime2025-01	integer	Find the sum of all integer bases $b>9$ for which $17_b$ is a
    4	GPQA Diamond	Physics	recoiTJPGUmzAkief	choice	Imagine an uncharged spherical conductor of radius $R$ having a
    5	SuperGPQA	Engineering	b7e20eac98764fb0bf30e8366d951daa	choice	Given a brick masonry compression member with $e / h=0.18$;
    6	AIME2025	Geometry	aime2025-16	integer	Six points $A, B, C, D, E$ and $F$ lie in a straight line in
    7	GPQA Diamond	Physics	rec4UqStf9

True

## 9. Tests automáticos sin modelo

Son los tests de `make test` que no necesitan cargar el modelo. Como no hay CUDA, se compilan con la versión
de CPU del motor. Cada uno comprueba una pieza concreta:

In [9]:
CPU_OBJS = "ds4_cpu.o ds4_image.o ds4_distributed.o ds4_tp.o ds4_ssd.o ds4_layer_pack.o ds4_engram.o"
MAKE_CPU = f'make -j$(nproc) CORE_OBJS="{CPU_OBJS}" DS4_LINK=cc DS4_LINK_LIBS="-lm -pthread"'

tests = [
    # (nombre, objetivo make o binario, explicación sencilla)
    ("Cuantización Q4_K",  "q4k-dot-test",
     "Que los pesos en formato Q4_K (4 bits) se decodifican y multiplican igual que una implementación de referencia"),
    ("Cuantización MXFP4", "mxfp4-dot-test",
     "Lo mismo para MXFP4, el formato de 4 bits en coma flotante de los expertos de DeepSeek"),
    ("Sampling (elegir la siguiente palabra)", "tests/test_sampling",
     "Que la elección del siguiente token (temperatura, top-p, min-p) da lo mismo que la versión de referencia, pero más rápido"),
    ("Sesiones y KV cache", "tests/test_session_state",
     "Gestión de la conversación: guardar y restaurar la KV cache, retroceder y reutilizar lo ya calculado"),
    ("Prefijo de conversación", "tests/test_prompt_prefix",
     "Lectura de ficheros con turnos USER:/ASSISTANT: para precargar una conversación"),
    ("Reparto de capas entre GPUs", "tests/test_layer_pack",
     "Reparto de las capas del modelo entre varias GPUs según la memoria de cada una (97 comprobaciones)"),
    ("Colocación multi-GPU del motor", "tests/test_engine_mgpu_placement",
     "Decisión de qué parte del modelo va a cada GPU y detección de cuándo no cabe (114 comprobaciones)"),
    ("Opciones --gpu-vram / --gpu-devices", "tests/test_gpu_args",
     "Lectura de las opciones --gpu-vram y --gpu-devices, rechazando valores inválidos"),
    ("Engram (tablas en disco)", "tests/test_engram",
     "Engram: tablas de DeepSeek V4.1 que se leen directamente del disco en vez de cargarse en RAM"),
    ("Lectura de memoria de Linux", "tests/test_linux_memory",
     "Cálculo de la RAM disponible en Linux (/proc/meminfo) antes de cargar el modelo"),
    ("Decodificar imágenes", "tests/test_image_decode",
     "Lectura de imágenes JPEG/PNG para los modelos con visión"),
    ("Troceado de imágenes (visión DeepSeek)", "tests/test_deepseek4_vision_image",
     "División de una imagen en fragmentos (tokens visuales) para el modelo con visión"),
    ("Agente de programación", "ds4_agent_test",
     "Herramientas del agente: leer/editar ficheros, ejecutar comandos e interpretar las llamadas a herramientas del modelo"),
    ("Red TCP para inferencia distribuida", "tests/test_tp_tcp",
     "Comunicación por red entre equipos en la inferencia distribuida: que no se pierden datos aunque un equipo se bloquee "
     "(los mensajes 'negotiation failed' son fallos provocados a propósito por el test)"),
    ("Comandos de inferencia distribuida", "tests/test_tp_commands",
     "Protocolo de mensajes entre equipos, incluidos los tiempos de espera (timeouts) y las desconexiones"),
]

for nombre, objetivo, explicacion in tests:
    binario = "./" + objetivo
    if objetivo.endswith("-test"):           # q4k-dot-test / mxfp4-dot-test compilan y ejecutan a la vez
        cmd = f"{MAKE_CPU} {objetivo}"
    else:
        cmd = f"{MAKE_CPU} {objetivo} > /tmp/build.log 2>&1 || (tail -5 /tmp/build.log; exit 1); {binario}"
    # test_tp_commands comprueba que un timeout de 50 ms se guarda EXACTO en el socket; algunos kernels de
    # Linux lo redondean (p. ej. a 52 ms) y el test falla aunque el código esté bien → lo marcamos como aviso.
    ejecutar(nombre, cmd, explicacion, mostrar_lineas=4,
             aviso_si="gate_timeout_ms' failed" if "tp_commands" in objetivo else None)

▶ Cuantización Q4_K
  $ make -j$(nproc) CORE_OBJS="ds4_cpu.o ds4_image.o ds4_distributed.o ds4_tp.o ds4_ssd.o ds4_layer_pack.o ds4_engram.o" DS4_LINK=cc DS4_LINK_LIBS="-lm -pthread" q4k-dot-test
  ℹ️  Que los pesos en formato Q4_K (4 bits) se decodifican y multiplican igual que una implementación de referencia
      dot known: result=2048.0 expected=2048.0: PASS
      dot vs reference (50 random blocks): PASS
    
    4/4 tests passed
  → ✅ OK  (0.2 s)

▶ Cuantización MXFP4
  $ make -j$(nproc) CORE_OBJS="ds4_cpu.o ds4_image.o ds4_distributed.o ds4_tp.o ds4_ssd.o ds4_layer_pack.o ds4_engram.o" DS4_LINK=cc DS4_LINK_LIBS="-lm -pthread" mxfp4-dot-test
  ℹ️  Lo mismo para MXFP4, el formato de 4 bits en coma flotante de los expertos de DeepSeek
      nibble order: -32.0 (expect -32.0): PASS
      dot vs independent reference: PASS
    
    4/4 tests passed
  → ✅ OK  (0.1 s)

▶ Sampling (elegir la siguiente palabra)
  $ make -j$(nproc) CORE_OBJS="ds4_cpu.o ds4_image.o ds4_distributed.o ds4_tp

> **Sobre el aviso ⚠️ en "Protocolo de mensajes entre equipos".** El test configura un tiempo de espera de 50 ms en
> una conexión y comprueba que al leerlo vale **exactamente** 50 ms. Pero Linux mide ese tiempo en "ticks" que,
> según el kernel, pueden ser de 4 ms, así que 50 ms se redondea a 52 ms. El código de `ds4` funciona; lo que
> falla es un test demasiado estricto para ciertos kernels. La celda siguiente lo demuestra:

In [10]:
%%bash
cat > /tmp/timeout.c <<'EOF'
#include <stdio.h>
#include <sys/socket.h>
#include <sys/time.h>
int main(void) {
    int fd[2]; socketpair(AF_UNIX, SOCK_STREAM, 0, fd);
    struct timeval pedido = {0, 50000}, leido; socklen_t n = sizeof leido;
    setsockopt(fd[0], SOL_SOCKET, SO_RCVTIMEO, &pedido, sizeof pedido);
    getsockopt(fd[0], SOL_SOCKET, SO_RCVTIMEO, &leido, &n);
    printf("Pido 50 ms de timeout y el kernel guarda %ld ms\n", leido.tv_sec*1000 + leido.tv_usec/1000);
}
EOF
cc /tmp/timeout.c -o /tmp/timeout && /tmp/timeout

Pido 50 ms de timeout y el kernel guarda 50 ms


## 10. Compilar para la GPU (solo si hay GPU)

Activa la GPU en `Entorno de ejecución → Cambiar tipo de entorno → GPU T4`. Esta celda compila `ds4` con CUDA
para la GPU asignada (la T4 tiene la arquitectura `sm_75`). Tarda **unos 5–10 minutos**, porque el código de GPU
es muy extenso.

Aun compilando, el modelo no se podrá ejecutar: la T4 tiene 15 GB de memoria y el modelo necesita unos 81 GiB.
Lo que demuestra es que el código de GPU es compatible con esta tarjeta.

In [11]:
gpu_cap = subprocess.run("nvidia-smi --query-gpu=compute_cap --format=csv,noheader", shell=True,
                         capture_output=True, text=True).stdout.strip().split("\n")[0]
if gpu_cap:
    arch = "sm_" + gpu_cap.replace(".", "")
    print(f"GPU detectada con arquitectura {arch}. Compilando en una copia aparte (varios minutos)...")
    !rm -rf /content/ds4-cuda && cp -r {REPO_DIR} /content/ds4-cuda && cd /content/ds4-cuda && make clean > /dev/null
    ejecutar(f"Compilar con CUDA ({arch})",
             f"cd /content/ds4-cuda && make -j$(nproc) ds4 CUDA_ARCH={arch} > /tmp/cuda.log 2>&1; r=$?; "
             f"grep -m5 -E 'error' /tmp/cuda.log; tail -2 /tmp/cuda.log; ls -lh ds4 2>/dev/null; exit $r",
             f"Que el código de GPU compila con nvcc para {arch}")
    if os.path.exists("/content/ds4-cuda/ds4"):
        ejecutar("`ds4` versión CUDA arranca", "cd /content/ds4-cuda && ./ds4 --help | head -3",
                 "Que el programa con CUDA arranca y encuentra las librerías de NVIDIA")
else:
    print("No hay GPU en este entorno: me salto esta parte.")

GPU detectada con arquitectura sm_75. Compilando en una copia aparte (varios minutos)...
▶ Compilar con CUDA (sm_75)
  $ cd /content/ds4-cuda && make -j$(nproc) ds4 CUDA_ARCH=sm_75 > /tmp/cuda.log 2>&1; r=$?; grep -m5 -E 'error' /tmp/cuda.log; tail -2 /tmp/cuda.log; ls -lh ds4 2>/dev/null; exit $r
  ℹ️  Que el código de GPU compila con nvcc para sm_75
    /usr/local/cuda/bin/nvcc -O3 -g -lineinfo --use_fast_math -arch=sm_75 -Xcompiler -march=native -Xcompiler -pthread -std=c++17 -c -o cuda/mmq/ds4_repack.o cuda/mmq/ds4_repack.cu
    /usr/local/cuda/bin/nvcc -O3 -g -lineinfo --use_fast_math -arch=sm_75 -Xcompiler -march=native -Xcompiler -pthread -o ds4 ds4_cli.o ds4_help.o ds4_prompt_prefix.o linenoise.o ds4_gpu_args.o ds4.o ds4_image.o ds4_distributed.o ds4_tp.o ds4_ssd.o ds4_cuda.o ds4_layer_pack.o ds4_engram.o cuda/mmq/ds4_ggml_stubs.o cuda/mmq/ds4_mmq.o cuda/mmq/ds4_mmq_d2r.o cuda/mmq/quantize.o cuda/mmq/mmid.o cuda/mmq/mmvq.o cuda/mmq/ds4_repack.o -lm -Xcompiler -pthread -L/usr/lo

## 11. Resultados

In [12]:
import pandas as pd
df = pd.DataFrame(RESULTADOS)
ok, aviso, fallo = (df["Resultado"].str.startswith(s).sum() for s in ("✅", "⚠️", "❌"))
commit = subprocess.run("git log -1 --format=%h", shell=True, cwd=REPO_DIR, capture_output=True, text=True).stdout.strip()

if fallo == 0:
    veredicto = "🟢 **Funciona**: compila, arranca y pasa los tests que no necesitan el modelo."
else:
    veredicto = "🔴 **Hay fallos**: revisa la salida de las pruebas marcadas con ❌."
display(Markdown(f"### {veredicto}\n\nCommit `{commit}` · ✅ {ok} · ⚠️ {aviso} · ❌ {fallo}"))
display(df.style.hide(axis="index").set_properties(**{"text-align": "left"}))

### 🟢 **Funciona**: compila, arranca y pasa los tests que no necesitan el modelo.

Commit `b3f5018` · ✅ 24 · ⚠️ 0 · ❌ 0

Prueba,Resultado,Qué comprueba,Tiempo (s)
Compilar con `make cpu`,✅ OK,Que el código compila y genera los 5 programas,61.100000
`ds4 --help`,✅ OK,Que el programa principal arranca y muestra sus opciones,0.000000
`ds4` sin modelo da error controlado,✅ OK,Que sin el modelo da un error claro en vez de bloquearse,0.000000
`ds4-server --help`,✅ OK,Que el servidor compatible con OpenAI/Anthropic arranca,0.000000
Listar preguntas de `ds4-eval`,✅ OK,Que el banco de preguntas está incluido y se puede listar,0.000000
Validar las preguntas de `ds4-eval`,✅ OK,Que todas las preguntas y respuestas esperadas están bien formadas,0.000000
Corrector automático de respuestas,✅ OK,Que se extrae bien la respuesta final del texto del modelo,0.000000
Cuantización Q4_K,✅ OK,Que los pesos en formato Q4_K (4 bits) se decodifican y multiplican igual que una implementación de referencia,0.200000
Cuantización MXFP4,✅ OK,"Lo mismo para MXFP4, el formato de 4 bits en coma flotante de los expertos de DeepSeek",0.100000
Sampling (elegir la siguiente palabra),✅ OK,"Que la elección del siguiente token (temperatura, top-p, min-p) da lo mismo que la versión de referencia, pero más rápido",32.800000


## 12. Conclusiones

- **Qué es.** Un motor de inferencia en C, especializado en unos pocos modelos MoE muy grandes. Frente a
  llama.cpp, que soporta muchos modelos, apuesta por pocos modelos muy optimizados.
- **Cómo lo consigue.** Combina tres ideas: **MoE** (de 284B parámetros solo trabajan ~13B por token),
  **cuantización agresiva** de los expertos (~2 bits por parámetro) y **atención comprimida** para contextos
  largos. Con SSD streaming e inferencia distribuida puede ir más allá de la memoria de un solo equipo.
- **Rendimiento publicado** (Flash Q2): un MacBook Pro M5 Max genera ~39 tokens/s y una DGX Spark ~18 tokens/s,
  ambos leyendo el mensaje de entrada a ~800 tokens/s.
- **¿Funciona?** El código compila sin errores, los programas arrancan y los tests que no necesitan el modelo
  pasan; la única excepción es un test demasiado estricto para algunos kernels de Linux. Lo que **no** se ha
  podido comprobar aquí es la generación de texto real, porque requiere el modelo (~81 GiB) y al menos
  96 GB de memoria.
- **Madurez.** El propio autor lo considera **beta**, y declara que se ha desarrollado con mucha ayuda de agentes
  de IA bajo supervisión humana.

### Uso con el hardware adecuado

```bash
git clone https://github.com/antirez/ds4.git && cd ds4
make                                  # Mac (Metal) · DGX Spark: make cuda-spark · Strix Halo: make strix-halo
./download_model.sh ds4f-q2           # DeepSeek V4 Flash Q2 (~81 GiB)
./ds4 -p "Explica qué es un modelo MoE en un párrafo."
./ds4-server --ctx 32768              # API compatible con OpenAI en http://127.0.0.1:8000
./ds4 --ssd-streaming --ctx 32768     # si el equipo tiene menos memoria
```